# UR5e pi0-FAST LoRA Fine-Tuning

**Run cells marked `[LAPTOP]` from your local machine before opening Colab.**  
**Run all other cells in Google Colab Pro (A100 recommended).**

## Requirements
- Google Colab Pro with A100 GPU (Runtime → Change runtime type → A100)
- HuggingFace account with write access to `sheilsarda/ur5_isaac_sim_v1`
- W&B account (wandb.ai)
- Dataset already converted with `convert_ur5_bag_to_lerobot.py`

---
## [LAPTOP] Push dataset to HuggingFace Hub

Run this cell **on your laptop** before starting the Colab session.  
It uploads `sheilsarda/ur5_isaac_sim_v1` from your local LeRobot cache to the Hub.

In [1]:
# [LAPTOP] — run this from your laptop, not Colab
#
# Prerequisites:
#   cd ~/Development/openpi && source .venv/bin/activate
#   huggingface-cli login   (paste your HF write token)
#
# Then run this cell, or paste the equivalent into a terminal:

import subprocess
result = subprocess.run([
    "python3", "-c",
    """
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
dataset = LeRobotDataset('sheilsarda/ur5_isaac_sim_v1')
dataset.push_to_hub(
    tags=['ur5e', 'isaac-sim'],
    private=False,
    push_videos=True,
    license='apache-2.0',
)
print('Done! Dataset live at https://huggingface.co/datasets/sheilsarda/ur5_isaac_sim_v1')
"""
], capture_output=False)


Done! Dataset live at https://huggingface.co/datasets/sheilsarda/ur5_isaac_sim_v1


---
## [COLAB] Everything below runs on Colab

Make sure you selected **A100** under Runtime → Change runtime type.

In [ ]:
# Cell 1: Verify GPU
!nvidia-smi

In [ ]:
# Cell 2: Clone repo and install dependencies
!git clone https://github.com/sheilsarda/openpi.git
%cd openpi
!pip install uv -q
!uv sync

In [ ]:
# Cell 3: Authenticate HuggingFace (needed to download dataset)
from huggingface_hub import login
login()  # paste your HF token (read access is sufficient)

In [ ]:
# Cell 4: Authenticate W&B
import wandb
wandb.login()  # paste your W&B API key

In [ ]:
# Cell 4b: Authenticate Google Cloud
# Required to download pretrained weights from gs://openpi-assets/
from google.colab import auth
auth.authenticate_user()

In [ ]:
# Cell 5: Mount Google Drive — checkpoints will be saved here
# so they survive the Colab session ending
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/openpi_checkpoints', exist_ok=True)

# Symlink so openpi writes checkpoints directly to Drive
if not os.path.exists('/content/openpi/checkpoints'):
    os.symlink('/content/drive/MyDrive/openpi_checkpoints', '/content/openpi/checkpoints')

print('Checkpoints will be saved to: /content/drive/MyDrive/openpi_checkpoints')

In [ ]:
# Cell 7: Train
#
# Config: pi0-FAST LoRA, action_horizon=10, max_token_len=180, batch_size=16
# Checkpoints saved every 1,000 steps to Google Drive
# W&B run: https://wandb.ai/sheilsarda/openpi
#
# Requires A100 (40GB). Expected time: ~1-2 hours for 5,000 steps.
!uv run scripts/train.py pi0_ur5 \
  --exp-name ur5_lift_v1 \
  --overwrite

---
## Head-to-Head: pi0-FAST vs pi0-base

Run one section at a time (or both sequentially). Both use LoRA, batch_size=16, 10k steps.

In [ ]:
# Norm stats — pi0-FAST (run once before training pi0_ur5)
!uv run scripts/compute_norm_stats.py --config-name=pi0_ur5

In [ ]:
# Train: pi0-FAST LoRA (pi0_ur5)
# Model: pi0-FAST + LoRA, action_horizon=10, max_token_len=180, batch_size=16
# W&B logs every 500 steps. Checkpoints → Google Drive every 1000 steps.
!uv run scripts/train.py pi0_ur5 \
  --exp-name ur5_fast_v1 \
  --overwrite

In [ ]:
# Norm stats — pi0-base (run once before training pi0_ur5_base)
!uv run scripts/compute_norm_stats.py --config-name=pi0_ur5_base

In [ ]:
# Train: pi0-base LoRA (pi0_ur5_base)
# Model: pi0 diffusion + LoRA, action_horizon=10, batch_size=16
# W&B logs every 500 steps. Checkpoints → Google Drive every 1000 steps.
!uv run scripts/train.py pi0_ur5_base \
  --exp-name ur5_base_v1 \
  --overwrite

In [ ]:
# Cell 8 (optional): Copy checkpoint back to HuggingFace for later use
# Run after training completes to persist weights outside of Drive
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path='/content/drive/MyDrive/openpi_checkpoints/pi0_ur5/ur5_lift_v1',
    repo_id='sheilsarda/pi0_ur5_lift_v1',
    repo_type='model',
    commit_message='pi0-FAST LoRA checkpoint after 5000 steps',
)